In [ ]:
import numpy as np


class GreyWolfOptimizer:
    """Grey Wolf Optimizer (GWO) Implementation.

    Based on Mirjalili et al. (2014) and provided algorithmic steps.
    """

    def __init__(self, obj_func, lb, ub, dim, n_wolves=12, max_iter=100):
        """Initialize GWO parameters.

        Parameters:
        - obj_func: Objective function to minimize
        - lb: Lower bound array or float
        - ub: Upper bound array or float
        - dim: Number of decision variables (dimension)
        - n_wolves: Population size (number of grey wolves)
        - max_iter: Maximum number of iterations
        """
        self.obj_func = obj_func
        self.dim = dim
        self.n_wolves = n_wolves
        self.max_iter = max_iter

        # Format lower and upper bounds as numpy arrays
        self.lb = np.array(lb) if isinstance(lb, list) else np.full(dim, lb)
        self.ub = np.array(ub) if isinstance(ub, list) else np.full(dim, ub)

    def optimize(self):
        # ------------------------------------------------------------------
        # Step 1 & 3: Population Initialization
        # Initialize positions of n wolves randomly within bounds
        # ------------------------------------------------------------------
        positions = np.random.uniform(
            low=self.lb, high=self.ub, size=(self.n_wolves, self.dim)
        )

        # Initialize Alpha, Beta, Delta positions and their fitness scores
        alpha_pos = np.zeros(self.dim)
        alpha_score = float("inf")

        beta_pos = np.zeros(self.dim)
        beta_score = float("inf")

        delta_pos = np.zeros(self.dim)
        delta_score = float("inf")

        # ------------------------------------------------------------------
        # Step 5 & 7: Evaluate initial population & Assign Alpha, Beta, Delta
        # ------------------------------------------------------------------
        for i in range(self.n_wolves):
            fitness = self.obj_func(positions[i])

            # Update Alpha, Beta, Delta based on fitness (minimization)
            if fitness < alpha_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = alpha_score, alpha_pos.copy()
                alpha_score, alpha_pos = fitness, positions[i].copy()
            elif fitness < beta_score:
                delta_score, delta_pos = beta_score, beta_pos.copy()
                beta_score, beta_pos = fitness, positions[i].copy()
            elif fitness < delta_score:
                delta_score, delta_pos = fitness, positions[i].copy()

        # Track history of best score over iterations
        convergence_curve = [alpha_score]

        # ------------------------------------------------------------------
        # Step 8: Main Optimization Loop
        # ------------------------------------------------------------------
        for t in range(self.max_iter):
            # Linearly decrease parameter 'a' from 2 to 0
            a = 2.0 - t * (2.0 / self.max_iter)

            # Update position of each search agent
            for i in range(self.n_wolves):
                for j in range(self.dim):
                    # --- Position update relative to Alpha ---
                    r1, r2 = np.random.rand(), np.random.rand()
                    A1 = 2 * a * r1 - a  # Equation (3.3 / Eq 8)
                    C1 = 2 * r2  # Equation (3.4 / Eq 9)

                    D_alpha = abs(
                        C1 * alpha_pos[j] - positions[i, j]
                    )  # Equation (3.5)
                    X1 = alpha_pos[j] - A1 * D_alpha  # Equation (3.6)

                    # --- Position update relative to Beta ---
                    r1, r2 = np.random.rand(), np.random.rand()
                    A2 = 2 * a * r1 - a
                    C2 = 2 * r2

                    D_beta = abs(C2 * beta_pos[j] - positions[i, j])
                    X2 = beta_pos[j] - A2 * D_beta

                    # --- Position update relative to Delta ---
                    r1, r2 = np.random.rand(), np.random.rand()
                    A3 = 2 * a * r1 - a
                    C3 = 2 * r2

                    D_delta = abs(C3 * delta_pos[j] - positions[i, j])
                    X3 = delta_pos[j] - A3 * D_delta

                    # Average of Alpha, Beta, Delta positions (Equation 3.7 / Eq 12)
                    positions[i, j] = (X1 + X2 + X3) / 3.0

                # Clip positions within lower and upper search bounds
                positions[i] = np.clip(positions[i], self.lb, self.ub)

                # ----------------------------------------------------------
                # Step 13: Evaluate fitness of updated wolf
                # ----------------------------------------------------------
                fitness = self.obj_func(positions[i])

                # ----------------------------------------------------------
                # Step 15: Update Alpha, Beta, Delta wolves
                # ----------------------------------------------------------
                if fitness < alpha_score:
                    delta_score, delta_pos = beta_score, beta_pos.copy()
                    beta_score, beta_pos = alpha_score, alpha_pos.copy()
                    alpha_score, alpha_pos = fitness, positions[i].copy()
                elif fitness < beta_score:
                    delta_score, delta_pos = beta_score, beta_pos.copy()
                    beta_score, beta_pos = fitness, positions[i].copy()
                elif fitness < delta_score:
                    delta_score, delta_pos = fitness, positions[i].copy()

            convergence_curve.append(alpha_score)

        # ------------------------------------------------------------------
        # Step 18: Produce and return the best solution found (Alpha)
        # ------------------------------------------------------------------
        return alpha_pos, alpha_score, convergence_curve


# =====================================================================
# Example Run: Minimizing the Korn function from slide 21
# f(x1, x2) = (x1 - 5)^2 + (x2 - 2)^2   ---> Global Minimum at (5, 2)
# =====================================================================
if __name__ == "__main__":

    def korn_function(x):
        return (x[0] - 5.0) ** 2 + (x[1] - 2.0) ** 2

    # Problem configuration
    dimensions = 2
    lower_bounds = [-10.0, -10.0]
    upper_bounds = [10.0, 10.0]
    num_wolves = 12
    iterations = 50

    # Instantiate and execute optimizer
    gwo = GreyWolfOptimizer(
        obj_func=korn_function,
        lb=lower_bounds,
        ub=upper_bounds,
        dim=dimensions,
        n_wolves=num_wolves,
        max_iter=iterations,
    )

    best_solution, best_fitness, curve = gwo.optimize()

    # Display results
    print("Optimization Complete!")
    print("-" * 40)
    print(f"Best Position Found (X_alpha) : {best_solution}")
    print(f"Best Fitness Value            : {best_fitness:.8f}")
    print(f"Theoretical Optimum           : [5.0, 2.0] with fitness 0.0")


Optimization Complete!
----------------------------------------
Best Position Found (X_alpha) : [5.01189488 2.00469397]
Best Fitness Value            : 0.00016352
Theoretical Optimum           : [5.0, 2.0] with fitness 0.0
